# HW1 Extraction Pipeline

Transcript -> minimum units -> FinBERT sentiment + LLM wins/risks/guidance -> call-level consensus & dispersion.

All business logic lives in `src/`; this notebook is pure orchestration. Stages are idempotent and cache to `data/cache/{units,sentiment,extractions,calls,qc}/` — re-running never redoes work that is already cached.

Phase rollout (plan section 1.8):

| Phase | Scope | Pass |
|---|---|---|
| P0 | `AMD_Q1-2024` | All stages green on 1 call |
| P1 | `AMD/NVDA/PLTR/JPM` Q1-2024 | QC gates green, LLM JSON fail ≤ 2% |
| P2 | 4 tickers × 5 transcripts | All QC gates green at scale |
| P3 | All 131 transcripts | Full corpus |

In [1]:
import sys
from pathlib import Path
from tqdm import tqdm

HW1_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(HW1_ROOT) not in sys.path:
    sys.path.insert(0, str(HW1_ROOT))

from src import aggregate, extract_batch, extract_llm, io_paths, parser, qc, sentiment_finbert

io_paths.ensure_dirs()
print("HW1 root:", HW1_ROOT)
print("transcripts:", len(io_paths.list_transcripts()))

HW1 root: /home/tian/code/baruch/NLP_HW/HW1
transcripts: 131


/home/tian/opt/miniconda/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Phase config

Pick the scope for this run. Default `PHASE = "P0"` runs just `AMD_Q1-2024` — flip to `P1` / `P2` / `P3` once each gate is green.

In [2]:
PHASE = "P3"
MODELS = ("gemma3:4b", "llama3.1:8b")

P1_CALLS = [("AMD", "Q1-2024"), ("NVDA", "Q1-2025"), ("PLTR", "Q1-2024"), ("JPM", "Q1-2024")]
P2_TICKERS = ["AMD", "NVDA", "JPM", "PLTR"]


def select_calls(phase: str) -> list[tuple[str, str]]:
    """Return `[(ticker, quarter), ...]` for the requested phase."""
    if phase == "P0":
        return [("AMD", "Q1-2024")]
    if phase == "P1":
        return list(P1_CALLS)
    if phase == "P2":
        calls: list[tuple[str, str]] = []
        for p in io_paths.list_transcripts():
            tic, q = io_paths.parse_stem(p)
            if tic in P2_TICKERS:
                calls.append((tic, q))
        calls.sort()
        seen: dict[str, int] = {}
        picked: list[tuple[str, str]] = []
        for tic, q in calls:
            seen[tic] = seen.get(tic, 0)
            if seen[tic] < 5:
                picked.append((tic, q))
                seen[tic] += 1
        return picked
    if phase == "P3":
        return [io_paths.parse_stem(p) for p in io_paths.list_transcripts()]
    raise ValueError(f"unknown phase: {phase}")


CALLS = select_calls(PHASE)
print(f"phase={PHASE}  calls={len(CALLS)}")
for c in CALLS[:8]:
    print(" ", c)
if len(CALLS) > 8:
    print(f"  ... and {len(CALLS) - 8} more")

phase=P3  calls=131
  ('AMD', 'Q1-2024')
  ('AMD', 'Q1-2025')
  ('AMD', 'Q2-2024')
  ('AMD', 'Q2-2025')
  ('AMD', 'Q3-2024')
  ('AMD', 'Q3-2025')
  ('AMD', 'Q4-2023')
  ('AMD', 'Q4-2024')
  ... and 123 more


## Stage A — Parser (transcript -> units)

Rule-based segmenter (Plan B). Writes `data/cache/units/<CALL>.jsonl`. We then evaluate the Stage-A QC gates (`executive_unknown_rate`, `qa_orphan_rate`, `parse_failures`) against the plan's targets.

In [3]:
all_units: dict[tuple[str, str], list[dict]] = {}
for tic, q in CALLS:
    path = io_paths.transcript_path(tic, q)
    units = parser.parse_file(path, tic, q)
    parser.write_units(units, tic, q)
    all_units[(tic, q)] = units

agg_units = [u for units in all_units.values() for u in units]
stage_a = qc.parse_quality(agg_units)
print("Stage A across selected calls:")
for k, v in stage_a.items():
    print(f"  {k}: {v}")
for name, passed, actual, tgt in qc.evaluate_gates(stage_a):
    mark = "PASS" if passed else "FAIL"
    print(f"  [{mark}] {name}: {actual:.3f} vs target {tgt}")
qc.save_qc_report(f"stage_a_{PHASE.lower()}", {"metrics": stage_a, "calls": [f"{t}_{q}" for t, q in CALLS]})

Stage A across selected calls:
  n_presenter: 431
  n_qa: 2908
  executive_unknown_rate: 0.0
  qa_orphan_rate: 0.009628610729023384
  parse_failures: 0
  [PASS] executive_unknown_rate: 0.000 vs target 0.05
  [PASS] qa_orphan_rate: 0.010 vs target 0.02
  [PASS] parse_failures: 0.000 vs target 0


PosixPath('/home/tian/code/baruch/NLP_HW/HW1/data/cache/qc/stage_a_p3.json')

## Stage B — FinBERT sentiment per unit

Scores every unit once. Cached at `data/cache/sentiment/<unit_id>.json`; re-running this cell after any scope change is cheap.

In [4]:
scorer = sentiment_finbert.FinBertScorer()
print(f"FinBERT device={scorer.device} dtype={scorer.dtype} batch_size={scorer.batch_size}")
sentiment_finbert.score_units(agg_units, scorer=scorer)

stage_b = qc.sentiment_quality(agg_units)
print("Stage B (FinBERT):", stage_b)
qc.save_qc_report(f"stage_b_{PHASE.lower()}", stage_b)

FinBERT device=cuda dtype=torch.bfloat16 batch_size=32
Stage B (FinBERT): {'n_seen': 3324, 'n_nan': 0, 'nan_rate': 0.0}


PosixPath('/home/tian/code/baruch/NLP_HW/HW1/data/cache/qc/stage_b_p3.json')

## Stage C — Ollama LLM extraction (wins / risks / guidance)

Two models in sequence (gemma3:4b, llama3.1:8b). Cached per (model, unit). Requires Ollama running at `http://localhost:11434`.

In [5]:
ok, detail = extract_llm.ollama_available(MODELS)
print("ollama:", ok, "|", detail)
if not ok:
    raise RuntimeError(
        "Ollama not reachable or models missing. Start `ollama serve` and `ollama pull` the required models."
    )

# Single-GPU tuning: iterate model OUTERMOST across the full corpus so each
# model is loaded exactly once (instead of swapping per call), and fan out
# requests with threads so Ollama's NUM_PARALLEL batching keeps the GPU busy.
extract_batch.extract_many(agg_units, models=MODELS, concurrency=4)

stage_c = qc.extraction_quality(agg_units, models=MODELS)
print("Stage C (LLM) per model:")
for m, rec in stage_c.items():
    rate = rec["failure_rate"]
    print(f"  {m}: n={rec['n_seen']} failed={rec['n_failed']} rate={rate}")
qc.save_qc_report(f"stage_c_{PHASE.lower()}", stage_c)

ollama: True | ok; 2 models


LLM llama3.1:8b  (431 pres + 342 qa-groups): 100%|█████████████████████████████████████████████████████████████| 773/773 [00:03<00:00, 197.02it/s]


Stage C (LLM) per model:
  gemma3:4b: n=838 failed=0 rate=0.0
  llama3.1:8b: n=830 failed=0 rate=0.0


PosixPath('/home/tian/code/baruch/NLP_HW/HW1/data/cache/qc/stage_c_p3.json')

## Stage D — Aggregate units -> call-level JSON

Pure-python roll-up from the caches above. Writes `data/cache/calls/<CALL>.json` with `consensus`, `dispersion`, `by_role`, and `by_section` tracks (Plan 1.7).

In [6]:
records = []
for tic, q in CALLS:
    rec = aggregate.build_and_write(tic, q, models=MODELS)
    records.append(rec)

sample = records[0]
print(f"{sample['ticker']} {sample['quarter']}  call_date={sample['call_date']}")
print("  n_units:", sample["n_units"])
print("  consensus.sentiment_call:", sample["consensus"].get("sentiment_call"))
print("  consensus.sentiment_qa:  ", sample["consensus"].get("sentiment_qa"))
print("  consensus.guidance_call: ", sample["consensus"].get("guidance_call"))
print("  consensus.wins_top5:")
for phrase, n in sample["consensus"].get("wins_top5", []):
    print(f"    {n:2d}x  {phrase}")
print("  consensus.risks_top5:")
for phrase, n in sample["consensus"].get("risks_top5", []):
    print(f"    {n:2d}x  {phrase}")
print("  dispersion.sentiment_dispersion:", sample["dispersion"].get("sentiment_dispersion"))
print("  dispersion.guidance_disagree:   ", sample["dispersion"].get("guidance_disagree"))
print("  dispersion.wins_overlap:        ", sample["dispersion"].get("wins_overlap"))
print("  dispersion.risks_overlap:       ", sample["dispersion"].get("risks_overlap"))

AMD Q1-2024  call_date=2024-04-30
  n_units: {'total': 31, 'presenter': 4, 'qa': 27}
  consensus.sentiment_call: 0.5022412986657845
  consensus.sentiment_qa:   0.39274098448249795
  consensus.guidance_call:  raised
  consensus.wins_top5:
     1x  earnings press release
     1x  accompanying slides
     1x  webcast
     1x  review a copy of our earnings press release and the accompanying slides
     1x  record data center GPU revenue
  consensus.risks_top5:
     1x  risks and uncertainties that could cause actual results to differ materially from our current expectations
     1x  factors that could cause actual results to differ materially
     1x  overall demand environment remain mixed
     1x  client segment revenue declined
     1x  embedded market conditions
  dispersion.sentiment_dispersion: 0.23200902773545828
  dispersion.guidance_disagree:    0
  dispersion.wins_overlap:         0.0
  dispersion.risks_overlap:        0.0


## Phase acceptance summary

In [7]:
summary = {
    "phase": PHASE,
    "n_calls": len(CALLS),
    "stage_a": stage_a,
    "stage_b": stage_b,
    "stage_c": stage_c,
}
qc.save_qc_report(f"summary_{PHASE.lower()}", summary)

gates: list[tuple[str, bool, float | int, float | int]] = []
gates.extend(qc.evaluate_gates(stage_a))
if stage_b.get("nan_rate") is not None:
    gates.extend(qc.evaluate_gates({"sentiment_nan_rate": stage_b["nan_rate"]}))
for m, rec in stage_c.items():
    if rec.get("failure_rate") is not None:
        gates.extend(
            [
                (f"llm_json_failure_rate[{m}]", rec["failure_rate"] <= qc.PASS_TARGETS["llm_json_failure_rate"],
                 rec["failure_rate"], qc.PASS_TARGETS["llm_json_failure_rate"])
            ]
        )

print(f"=== {PHASE} gate summary ===")
all_pass = True
for name, passed, actual, tgt in gates:
    mark = "PASS" if passed else "FAIL"
    all_pass &= passed
    print(f"  [{mark}] {name}: {actual} vs <= {tgt}")
print(f"=== overall: {'PASS' if all_pass else 'FAIL'} ===")

=== P3 gate summary ===
  [PASS] executive_unknown_rate: 0.0 vs <= 0.05
  [PASS] qa_orphan_rate: 0.009628610729023384 vs <= 0.02
  [PASS] parse_failures: 0 vs <= 0
  [PASS] sentiment_nan_rate: 0.0 vs <= 0.05
  [PASS] llm_json_failure_rate[gemma3:4b]: 0.0 vs <= 0.02
  [PASS] llm_json_failure_rate[llama3.1:8b]: 0.0 vs <= 0.02
=== overall: PASS ===
